# Headline Mood Check (MVP)

**Track:** Finance · **Team:** Jinwon Song, Sangjun Ha

A beginner investor pastes the headlines they just read and gets a card that shows how negative the news is,
a model's estimate for the Dow's next trading day, and, right next to it, how that model did on two years it never saw.

> This project is a prototype and is not financial advice.

**How this notebook is organised**

| Part | What it does | Data it may use |
|---|---|---|
| 0 | Load the Kaggle files | - |
| 1 | Split the data by date | all |
| 2 | Build: FinBERT scores, daily features, LightGBM | **ML set only** (2008-08-08 to 2014-12-31) |
| 3 | Test once | **Test set** (2015-01-02 to 2016-07-01) |
| 4 | The tool: paste headlines, or pick a test-period date | saved files from Parts 2 and 3 |
| A | Appendix: our 40 labels vs FinBERT, small language model trial | ML set |

Run the cells from top to bottom on a GPU runtime (Runtime > Change runtime type > T4 GPU). Nothing has to be uploaded: the data comes from the public Kaggle page and our label file from this project's GitHub repository. If a download fails, the notebook asks for the file instead. Scoring all headlines with FinBERT takes about 8 minutes.

## Part 0. Load the data

In [ ]:
import os, re, glob, json, time, zipfile, shutil
import numpy as np
import pandas as pd

DATA_DIR = "/content"
OUT = os.path.join(DATA_DIR, "mvp")          # everything this notebook saves goes here
os.makedirs(OUT, exist_ok=True)
KAGGLE_DATASET = "goodenoughzhenlei/combined-news-djiacsv"   # the public Kaggle page cited in the README

def unzip_uploads():
    for z in glob.glob(os.path.join(DATA_DIR, "*.zip")):
        with zipfile.ZipFile(z) as f:
            f.extractall(DATA_DIR)

def find(pattern):
    hits = sorted(glob.glob(os.path.join(DATA_DIR, pattern)), key=os.path.getmtime, reverse=True)
    return hits[0] if hits else None

def have_data():
    return find("Combined_News_DJIA*.csv") is not None and find("*DJIA_table*.csv") is not None

def download_from_kaggle():
    """Copy the two data files from the public Kaggle page into this runtime (no upload needed)."""
    try:
        import kagglehub
        folder = kagglehub.dataset_download(KAGGLE_DATASET)
        for p in glob.glob(os.path.join(folder, "**", "*"), recursive=True):
            if p.lower().endswith((".csv", ".zip")):
                shutil.copy(p, DATA_DIR)
        unzip_uploads()
    except Exception as e:
        print("Automatic download did not work:", type(e).__name__, str(e)[:200])

unzip_uploads()
source = "files already in this runtime"
if not have_data():
    download_from_kaggle()
    source = "downloaded from Kaggle"
if not have_data():
    from google.colab import files
    print("Upload Combined_News_DJIA.csv (or its .zip) and DJIA_table.csv from the Kaggle page.")
    files.upload()
    unzip_uploads()
    source = "uploaded by hand"
print("Data source:", source)

news = pd.read_csv(find("Combined_News_DJIA*.csv"))
price = pd.read_csv(find("*DJIA_table*.csv"))
SCORE_CACHE = find("finbert_scores*.csv")    # optional: skips about 8 minutes of FinBERT
print("News file:", news.shape, "| price file:", price.shape, "| FinBERT cache:", os.path.basename(SCORE_CACHE) if SCORE_CACHE else "none")

## Part 1. Split the data by date (never at random)

* **ML set** (2008-08-08 to 2014-12-31): used to build the model. Inside it, 2008-2013 trains and 2014 checks the settings once.
* **Test set** (2015-01-02 to 2016-07-01): kept closed until Part 3 and used once.

In [ ]:
TOP_COLS = [f"Top{i}" for i in range(1, 26)]
WRAP = re.compile(r"""^b(['"])(.*)\1$""", re.S)

def clean(text):
    if pd.isna(text):
        return ""
    text = str(text).strip()
    m = WRAP.match(text)
    if m:
        text = m.group(2)
    return text.replace("\\'", "'").replace('\\"', '"').strip()

news["Date"] = pd.to_datetime(news["Date"]); price["Date"] = pd.to_datetime(price["Date"])
news = news.sort_values("Date").reset_index(drop=True)
price = price.sort_values("Date").reset_index(drop=True)
for c in TOP_COLS:
    news[c] = news[c].map(clean)
price["ret"] = price["Adj Close"].pct_change()                     # daily return
news = news.merge(price[["Date", "Adj Close", "ret"]], on="Date", how="left")

ML_END, TRAIN_END = pd.Timestamp("2014-12-31"), pd.Timestamp("2013-12-31")
news["part"] = np.where(news.Date <= TRAIN_END, "train", np.where(news.Date <= ML_END, "valid", "test"))

ml_set = news[news.Date <= ML_END].reset_index(drop=True)
test_set = news[news.Date > ML_END].reset_index(drop=True)
ml_set.to_csv(os.path.join(OUT, "ml_set.csv"), index=False)
test_set.to_csv(os.path.join(OUT, "test_set.csv"), index=False)

for name, part in [("ML set: train", ml_set[ml_set.part == "train"]), ("ML set: valid", ml_set[ml_set.part == "valid"]),
                   ("Test set", test_set)]:
    print(f"{name:14s} {len(part):4d} days  {part.Date.min().date()} to {part.Date.max().date()}  up rate {part.Label.mean():.3f}")
print("Days without a price match:", int(news["Adj Close"].isna().sum()))
print(f"Label matches 'adjusted close >= previous day': {((news.ret >= 0).astype(int) == news.Label)[1:].mean():.3f}")

## Part 2. Build the model (ML set only)

**Step 2.1 FinBERT.** Each headline gets `score = P(positive) - P(negative)` from `ProsusAI/finbert`
(CC BY-NC-SA 4.0, used unmodified and loaded from Hugging Face, weights not redistributed).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
FINBERT = "ProsusAI/finbert"
_finbert = {}

def load_finbert():
    if not _finbert:
        tok = AutoTokenizer.from_pretrained(FINBERT)
        mdl = AutoModelForSequenceClassification.from_pretrained(FINBERT).to(device).eval()
        id2label = {i: l.lower() for i, l in mdl.config.id2label.items()}
        _finbert.update(tok=tok, mdl=mdl, idx={l: i for i, l in id2label.items()})
    return _finbert

def finbert_score(texts, batch_size=64):
    fb = load_finbert()
    out = []
    for i in range(0, len(texts), batch_size):
        enc = fb["tok"](texts[i:i + batch_size], padding=True, truncation=True, max_length=96, return_tensors="pt").to(device)
        with torch.no_grad():
            out.append(torch.softmax(fb["mdl"](**enc).logits, dim=-1).cpu().numpy())
    p = np.vstack(out)
    res = pd.DataFrame({"p_pos": p[:, fb["idx"]["positive"]], "p_neg": p[:, fb["idx"]["negative"]],
                        "p_neu": p[:, fb["idx"]["neutral"]]})
    res["score"] = res.p_pos - res.p_neg
    res["finbert_label"] = res[["p_pos", "p_neg", "p_neu"]].values.argmax(1)
    res["finbert_label"] = res.finbert_label.map({0: "positive", 1: "negative", 2: "neutral"})
    return res

def to_long(frame):
    long = frame.melt(id_vars=["Date"], value_vars=TOP_COLS, var_name="rank", value_name="text")
    long["rank"] = long["rank"].str.replace("Top", "").astype(int)
    return long[long.text != ""].sort_values(["Date", "rank"]).reset_index(drop=True)

def scores_for(frame):
    long = to_long(frame)
    if SCORE_CACHE:
        cache = pd.read_csv(SCORE_CACHE, parse_dates=["Date"])[["Date", "rank", "p_pos", "p_neg", "p_neu", "score", "finbert_label"]]
        got = long.merge(cache, on=["Date", "rank"], how="left")
        if got.score.notna().all():
            return got
        print("Cache incomplete, scoring with FinBERT instead")
    t0 = time.time()
    got = pd.concat([long, finbert_score(long.text.tolist())], axis=1)
    print(f"FinBERT scored {len(got)} headlines in {(time.time() - t0) / 60:.1f} min")
    return got

ml_scores = scores_for(ml_set)
print("ML-set headlines scored:", len(ml_scores))
print("FinBERT labels in the ML set:", ml_scores.finbert_label.value_counts(normalize=True).round(3).to_dict())

**Step 2.2 Daily features (8).** Mean, minimum, maximum and spread of the day's scores, share of negative and positive
headlines, 5-day average mood, and change from the previous day.

* Problem A (the brief): day *t* headlines → day *t* direction
* Problem B (our question, used by the tool): day *t-1* headlines → day *t* direction, which is what a person could know beforehand

In [ ]:
FEATS = ["s_mean", "s_min", "s_max", "s_std", "neg_share", "pos_share", "s_mean_ma5", "s_mean_chg"]

def day_stats(scored):
    g = scored.groupby("Date")
    return pd.DataFrame({
        "s_mean": g.score.mean(), "s_min": g.score.min(), "s_max": g.score.max(), "s_std": g.score.std(),
        "neg_share": g.finbert_label.apply(lambda s: (s == "negative").mean()),
        "pos_share": g.finbert_label.apply(lambda s: (s == "positive").mean()),
    }).reset_index()

def daily_table(frame, scored):
    d = frame[["Date", "Label", "ret", "part"]].merge(day_stats(scored), on="Date", how="left").sort_values("Date").reset_index(drop=True)
    d["s_mean_ma5"] = d.s_mean.rolling(5, min_periods=1).mean()
    d["s_mean_chg"] = d.s_mean.diff()
    return d

def problem_A(d):
    return d[["Date", "part", "Label", "ret"] + FEATS].copy()

def problem_B(d):
    b = d[["Date", "part", "Label", "ret"]].copy()
    b[FEATS] = d[FEATS].shift(1)
    return b.iloc[1:].reset_index(drop=True)

ml_daily = daily_table(ml_set, ml_scores)
A_ml, B_ml = problem_A(ml_daily), problem_B(ml_daily)
print("ML-set days:", len(ml_daily), "| problem A rows:", len(A_ml), "| problem B rows:", len(B_ml))

**Step 2.3 LightGBM.** A small, strongly regularised model, because there are only about 1,600 days and the signal is weak.
Settings are fixed here and not changed after seeing any result. We train on 2008-2013, check on 2014 once,
then retrain on the whole ML set and save the models.

In [ ]:
import lightgbm as lgb
from math import sqrt

PARAMS = dict(n_estimators=200, learning_rate=0.03, num_leaves=4, min_child_samples=40,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=5.0, verbose=-1)
SEEDS = [42, 43, 44, 45, 46]

def fit(frame, seed):
    m = lgb.LGBMClassifier(**PARAMS, random_state=seed)
    m.fit(frame[FEATS], frame.Label)
    return m

def acc_ci(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    acc = (y_true == y_pred).mean()
    half = 1.96 * sqrt(acc * (1 - acc) / len(y_true))
    return acc, acc - half, acc + half

valid_rows = []
models = {}
for name, frame in [("A", A_ml), ("B", B_ml)]:
    tr, va = frame[frame.part == "train"], frame[frame.part == "valid"]
    accs = [(fit(tr, s).predict(va[FEATS]) == va.Label).mean() for s in SEEDS]
    valid_rows.append({"model": f"LightGBM, problem {name}", "valid 2014 (mean of 5 seeds)": round(np.mean(accs), 3)})
    models[name] = {s: fit(frame, s) for s in SEEDS}          # final models: whole ML set
va = ml_daily[ml_daily.part == "valid"]
valid_rows += [{"model": "Baseline: always up", "valid 2014 (mean of 5 seeds)": round(va.Label.mean(), 3)},
               {"model": "Baseline: same as previous day",
                "valid 2014 (mean of 5 seeds)": round((va.Label == ml_daily.Label.shift(1)[va.index]).mean(), 3)}]
print(pd.DataFrame(valid_rows).to_string(index=False))

# Save what the tool needs (Part 4 reads only these files)
models["B"][42].booster_.save_model(os.path.join(OUT, "model_B.txt"))
ref = {"feats": FEATS, "params": {k: v for k, v in PARAMS.items()}, "seeds": SEEDS,
       "ref_s_mean": ml_daily.s_mean.dropna().round(6).tolist(),
       "mood_fifth_cuts": ml_daily.s_mean.quantile([0.2, 0.4, 0.6, 0.8]).round(6).tolist()}
json.dump(ref, open(os.path.join(OUT, "reference_2008_2014.json"), "w"))
print("Saved model_B.txt and reference_2008_2014.json")

## Part 3. Test once (test set)

Everything below uses 2015-01-02 to 2016-07-01 for the first time.
The 5-day average for the first test days uses headlines from late 2014, which were public before those days (no labels are used).

In [ ]:
test_scores = scores_for(test_set)
all_scores = pd.concat([ml_scores, test_scores], ignore_index=True)
all_frame = pd.concat([ml_set, test_set], ignore_index=True)
daily = daily_table(all_frame, all_scores)                      # rolling features cross the boundary, labels do not
A_all, B_all = problem_A(daily), problem_B(daily)
A_te, B_te = A_all[A_all.part == "test"], B_all[B_all.part == "test"]

rows = []
for name, frame in [("A (same-day headlines)", A_te), ("B (previous-day headlines)", B_te)]:
    key = name[0]
    accs = [(models[key][s].predict(frame[FEATS]) == frame.Label).mean() for s in SEEDS]
    pred42 = models[key][42].predict(frame[FEATS])
    acc, lo, hi = acc_ci(frame.Label, pred42)
    rows.append({"model": f"LightGBM, problem {name}", "test accuracy (mean of 5 seeds)": round(np.mean(accs), 3),
                 "seed range": f"{min(accs):.3f} to {max(accs):.3f}", "95% CI (seed 42)": f"{lo:.3f} to {hi:.3f}",
                 "days predicted up": round(pred42.mean(), 3)})
te = daily[daily.part == "test"]
for name, pred in [("Baseline: always up", np.ones(len(te), dtype=int)),
                   ("Baseline: same as previous day", daily.Label.shift(1)[te.index].astype(int))]:
    acc, lo, hi = acc_ci(te.Label, pred)
    rows.append({"model": name, "test accuracy (mean of 5 seeds)": round(acc, 3), "seed range": "-",
                 "95% CI (seed 42)": f"{lo:.3f} to {hi:.3f}", "days predicted up": 1.0 if "up" in name else "-"})
test_table = pd.DataFrame(rows)
print(f"Test days: {len(te)}")
print(test_table.to_string(index=False))

report = {"test_days": int(len(te)),
          "model_B_acc": float(test_table.iloc[1]["test accuracy (mean of 5 seeds)"]),
          "always_up_acc": float(test_table.iloc[2]["test accuracy (mean of 5 seeds)"]),
          "prev_day_acc": float(test_table.iloc[3]["test accuracy (mean of 5 seeds)"]),
          "margin_pts": round(100 * 1.96 * sqrt(0.25 / len(te)), 1)}
json.dump(report, open(os.path.join(OUT, "test_report.json"), "w"))
print("Saved test_report.json")

**Mood and the size of moves.** Rule set before looking: a relationship counts only if it points the same way
in both periods and has p < 0.05 in both.

In [ ]:
from scipy.stats import spearmanr

d = daily.copy()
d["ret_next"], d["label_next"] = d.ret.shift(-1), d.Label.shift(-1)
d["abs_ret"], d["abs_ret_next"] = d.ret.abs(), d.ret_next.abs()
d["period"] = np.where(d.part == "test", "test 2015-16", "ML set 2008-14")

rows = []
for period, part in d.groupby("period", sort=False):
    for x, y, meaning in [("s_mean", "ret", "mood vs same-day return"),
                          ("s_mean", "ret_next", "mood vs next-day return"),
                          ("neg_share", "abs_ret", "negative share vs size of same-day move"),
                          ("neg_share", "abs_ret_next", "negative share vs size of next-day move")]:
        ok = part[[x, y]].dropna()
        rho, p = spearmanr(ok[x], ok[y])
        rows.append({"period": period, "relation": meaning, "days": len(ok), "rho": round(rho, 3), "p": round(p, 3)})
corr = pd.DataFrame(rows)
wide = corr.pivot(index="relation", columns="period", values=["rho", "p"])
passes = [(np.sign(wide.loc[r, ("rho", "ML set 2008-14")]) == np.sign(wide.loc[r, ("rho", "test 2015-16")]))
          and (wide.loc[r, ("p", "ML set 2008-14")] < 0.05) and (wide.loc[r, ("p", "test 2015-16")] < 0.05) for r in wide.index]
wide["passes rule"] = passes
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
print(wide)

d["mood_fifth"] = np.digitize(d.s_mean, ref["mood_fifth_cuts"]) + 1
print("\nMood fifths (1 = most negative days); cut points from the ML set only:")
print(d.groupby(["period", "mood_fifth"]).agg(days=("label_next", "size"), next_day_up_rate=("label_next", "mean"),
      mean_next_day_return_pct=("ret_next", lambda s: 100 * s.mean())).round(3))

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.plot(d.Date, all_frame.sort_values("Date")["Adj Close"].values, color="#4a4a4a", lw=1)
ax1.set_ylabel("DJIA (adjusted close)")
ax2 = ax1.twinx()
ax2.plot(d.Date, d.s_mean.rolling(20, min_periods=5).mean(), color="#c0392b", lw=1)
ax2.set_ylabel("News mood, 20-day average (FinBERT)")
ax1.axvline(pd.Timestamp("2015-01-01"), color="#888", ls="--", lw=1)
ax1.text(pd.Timestamp("2015-02-01"), ax1.get_ylim()[0], " test period", fontsize=9, va="bottom", color="#555")
ax1.set_title("DJIA and the mood of the day's top 25 world-news headlines")
fig.tight_layout()
fig.savefig(os.path.join(OUT, "mood_vs_djia.png"), dpi=150)
plt.show()

# Where it fails: the test days where the model was most confident and wrong
tb = B_te.copy()
tb["prob_up"] = models["B"][42].predict_proba(tb[FEATS])[:, 1]
wrong = tb[(tb.prob_up >= 0.5).astype(int) != tb.Label].assign(conf=lambda f: (f.prob_up - 0.5).abs()).nlargest(2, "conf")
print("Most confident wrong calls (problem B):")
for _, r in wrong.iterrows():
    prev_day = daily.Date[daily.Date < r.Date].max()
    print(f"\n{r.Date.date()}: estimate {100 * r.prob_up:.0f}% up, actual {'up' if r.Label else 'down'} ({100 * r.ret:+.2f}%). "
          f"Headlines from {prev_day.date()}, most negative first:")
    for t in all_scores[all_scores.Date == prev_day].sort_values("score").text.head(5):
        print("   -", t[:120])

## Part 4. The tool (MVP)

The tool reads only the files saved in Parts 2 and 3: `model_B.txt`, `reference_2008_2014.json`, `test_report.json`.

**4a. Paste the headlines you just read** (one per line, English). Then run the cell.

In [ ]:
TOOL = {"model": lgb.Booster(model_file=os.path.join(OUT, "model_B.txt")),
        "ref": json.load(open(os.path.join(OUT, "reference_2008_2014.json"))),
        "report": json.load(open(os.path.join(OUT, "test_report.json")))}
NICE = {"s_mean": "average headline mood", "s_min": "most negative headline", "s_max": "most positive headline",
        "s_std": "how mixed the headlines were", "neg_share": "share of negative headlines",
        "pos_share": "share of positive headlines", "s_mean_ma5": "5-day average mood",
        "s_mean_chg": "change in mood from the previous day"}

def card(scored, feats, title, notes=(), looking_back=None):
    ref, rep, booster = TOOL["ref"], TOOL["report"], TOOL["model"]
    x = pd.DataFrame([feats])[ref["feats"]]
    prob_up = float(booster.predict(x)[0])
    contrib = pd.Series(booster.predict(x, pred_contrib=True)[0][:-1], index=ref["feats"])
    more_neg_than = round(100 * np.mean(np.array(ref["ref_s_mean"]) > feats["s_mean"]))
    counts = scored.finbert_label.value_counts()
    s = scored.sort_values("score")
    print(f"=== Headline Mood Check: {title} ===")
    print(f"1. News mood: more negative than {more_neg_than}% of days in 2008-2014 "
          f"(FinBERT: {counts.get('negative', 0)} negative, {counts.get('neutral', 0)} neutral, {counts.get('positive', 0)} positive).")
    print("   Almost no world-news headline reads as positive, so compare days with each other, not with zero.")
    print("2. Most negative headlines:")
    for t in s.text.head(3): print("   -", t[:120])
    print("   Most positive headlines:")
    for t in s.text.tail(3)[::-1]: print("   -", t[:120])
    print(f"3. Model estimate that the Dow closes up on the next trading day: {round(100 * prob_up)}%")
    for f in contrib.abs().sort_values(ascending=False).index[:2]:
        print(f"   - {NICE[f]} pushed the estimate {'up' if contrib[f] > 0 else 'down'}")
    print(f"4. Track record on 2015-16 ({rep['test_days']} days, about +/-{rep['margin_pts']:.0f} points of chance):")
    print(f"   this model {100 * rep['model_B_acc']:.1f}% | always guess 'up' {100 * rep['always_up_acc']:.1f}% | "
          f"same as previous day {100 * rep['prev_day_acc']:.1f}%")
    verdict = "did not beat" if rep["model_B_acc"] <= rep["always_up_acc"] else "was within chance of"
    print(f"5. News mood alone {verdict} simply guessing 'up' in 2015-16 "
          f"({100 * rep['model_B_acc']:.1f}% vs {100 * rep['always_up_acc']:.1f}%), so this estimate is not a reason to trade on its own.")
    for n in notes: print("   Note:", n)
    if looking_back: print("  ", looking_back)
    print("6. This project is a prototype and is not financial advice.")
    return prob_up

def mood_check(headlines_text):
    heads = [h.strip() for h in headlines_text.strip().splitlines() if h.strip()]
    if len(heads) < 5:
        raise ValueError("Paste at least 5 headlines, one per line.")
    scored = pd.concat([pd.DataFrame({"text": heads}), finbert_score(heads)], axis=1)
    st = day_stats(scored.assign(Date=0)).iloc[0].to_dict()
    feats = {**st, "s_mean_ma5": st["s_mean"], "s_mean_chg": 0.0}
    notes = ["Only one day of headlines was given, so the 5-day average is set to today's mood and the change to zero.",
             "The model learned from 2008-2014 world news; today's news may behave differently."]
    if len(heads) != 25:
        notes.append(f"The 2008-2014 days had 25 headlines each; you gave {len(heads)}, so compare with care.")
    return card(scored, feats, f"your {len(heads)} headlines", notes)

# Replace these example lines with headlines you read today (these are from the dataset, 2016-06-24).
MY_HEADLINES = """
""" + "\n".join(test_set.loc[test_set.Date == "2016-06-24", TOP_COLS[:8]].iloc[0].tolist()) + """
"""
_ = mood_check(MY_HEADLINES)

**4b. Pick a test-period date** (demo). The card uses that day's real 25 headlines. The last line looks back at what happened, which a user could not know at the time.

In [ ]:
def mood_check_date(date_str):
    day = pd.Timestamp(date_str)
    row = daily[(daily.Date == day) & (daily.part == "test")]
    if row.empty or row.index[0] + 1 >= len(daily):
        raise ValueError("Pick a trading day between 2015-01-02 and 2016-06-30.")
    i = row.index[0]
    feats = row.iloc[0][FEATS].to_dict()
    nxt = daily.loc[i + 1]
    back = f"(Looking back: on {nxt.Date.date()} the Dow actually went {'up' if nxt.Label else 'down'}, {100 * nxt.ret:+.2f}%.)"
    card(all_scores[all_scores.Date == day], feats, day.date().isoformat(), looking_back=back)

mood_check_date("2015-08-21")      # the day before the Aug 24, 2015 drop
print()
plain_day = daily[daily.part == "test"].Date.iloc[:-1].sample(1, random_state=7).iloc[0]
mood_check_date(plain_day.date().isoformat())   # an ordinary day picked with a fixed seed

## Appendix A. Our 40 labels vs FinBERT (ML set only)

40 headlines drawn with a fixed seed from 2008-2014, labelled by Sangjun Ha with help from ChatGPT before we saw FinBERT's labels for them.
The label file is read from the repository's `data/` folder.

In [ ]:
from sklearn.metrics import cohen_kappa_score

LABEL_URL = "https://raw.githubusercontent.com/extraolive7737/VTAI-Hackathon.team.Extraolive/main/data/headline_labels_40.csv"

def filled_label_file():
    paths = sorted(glob.glob(os.path.join(DATA_DIR, "headline_labels_40*.csv")), key=os.path.getmtime, reverse=True)
    for p in paths:
        if pd.read_csv(p).our_label.notna().any():
            return p
    return None

LABEL_FILE = filled_label_file()
if LABEL_FILE is None:
    try:                                   # our own label file, from this project's GitHub repository
        pd.read_csv(LABEL_URL).to_csv(os.path.join(DATA_DIR, "headline_labels_40_repo.csv"), index=False)
    except Exception as e:
        print("Could not download the label file:", type(e).__name__)
        from google.colab import files
        files.upload()
    LABEL_FILE = filled_label_file()
print("Label file:", os.path.basename(LABEL_FILE))
lab = pd.read_csv(LABEL_FILE, parse_dates=["Date"])
lab["our_label"] = lab.our_label.astype(str).str.strip().str.lower()
lab = lab.merge(ml_scores[["Date", "rank", "finbert_label"]], on=["Date", "rank"], how="left")
assert lab.finbert_label.notna().all() and lab.our_label.isin(["positive", "negative", "neutral"]).all()
print(f"Headlines: {len(lab)} | agreement: {(lab.our_label == lab.finbert_label).mean():.1%} | "
      f"Cohen's kappa: {cohen_kappa_score(lab.our_label, lab.finbert_label):.2f}")
print(pd.crosstab(lab.our_label, lab.finbert_label, rownames=["ours"], colnames=["FinBERT"]))

## Appendix B. Small language model trial (not used in the tool)

We tried `Qwen/Qwen2.5-1.5B-Instruct` (Apache 2.0) to write the card's explanation, with an automatic check for numbers,
advice words and sentence length, on three separate sets of 20 test days. The full runs and outputs are in our
experiment notebook (linked in the README). Recorded results:

| Try | Passed the automatic check | Usable after a stricter check |
|---|---|---|
| 1. Explain the card's numbers | 8 of 20 | not re-checked |
| 2. Same, revised prompt | 13 of 20 | 0 of 20 (all 13 cut off mid-sentence) |
| 3. Name the day's topics, no numbers | 0 of 20 | - |

Errors the check could not catch: "57% chance of an up day" became "correct 57% of the time", a 66% estimate became
"about $66 higher", and claims became facts ("Missing MH370 found"). The card therefore uses a fixed template sentence.